In [4]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

# Membaca dataset.
data = pd.read_csv(r"C:\Users\MSI1\Downloads\New\Sample - Superstore.csv",
                   encoding='cp1252', low_memory=False)
df = pd.DataFrame(data)

# Buat copy untuk memperbaiki data
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,1/21/2014,1/23/2014,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,33180,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028
9990,9991,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332
9991,9992,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932
9992,9993,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200


In [6]:
# print(df.info())

print(df.dtypes)

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object


In [7]:
missing_values = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

missing_values = missing_values[missing_values["Missing Count"] > 0]

print(missing_values)

Empty DataFrame
Columns: [Column, Missing Count, Missing Percentage]
Index: []


In [8]:
duplicate_count = df.duplicated().sum()
print("Jumlah data duplicate:", duplicate_count)

Jumlah data duplicate: 0


In [9]:
print(df.columns)

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')


In [10]:
text_cols = df.select_dtypes(include=["object"]).columns

for col in text_cols:
    inconsistent_spaces = df[df[col].astype(str) != df[col].astype(str).str.strip()]
    
    if len(inconsistent_spaces) > 0:
        print(f"Kolom '{col}' memiliki spasi tidak konsisten: {len(inconsistent_spaces)} data")

Kolom 'Product Name' memiliki spasi tidak konsisten: 16 data


In [11]:
print("\n===== OUTLIER CHECK USING IQR =====")
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

outlier_result = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outlier_count = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    
    outlier_result.append([
        col, Q1, Q3, IQR, lower, upper, outlier_count
    ])

outlier_df = pd.DataFrame(
    outlier_result,
    columns=["Column", "Q1", "Q3", "IQR", "Lower Bound", "Upper Bound", "Outlier Count"]
)

print(outlier_df)


===== OUTLIER CHECK USING IQR =====
        Column           Q1         Q3          IQR   Lower Bound  \
0       Row ID   2499.25000   7495.750   4996.50000  -4995.500000   
1  Postal Code  23223.00000  90008.000  66785.00000 -76954.500000   
2        Sales     17.28000    209.940    192.66000   -271.710000   
3     Quantity      2.00000      5.000      3.00000     -2.500000   
4     Discount      0.00000      0.200      0.20000     -0.300000   
5       Profit      1.72875     29.364     27.63525    -39.724125   

     Upper Bound  Outlier Count  
0   14990.500000              0  
1  190185.500000              0  
2     498.930000           1167  
3       9.500000            170  
4       0.500000            856  
5      70.816875           1881  
